In [31]:
# Import necessary libraries
import requests
from tqdm import tqdm
import re
import pandas as pd 
import json
import ast

In [ ]:
# Fill these variables
endpoint = "https://...."
deployment_id = "#-gpt-5"
api_version = "2025-04-01-preview" 
api_key = "#"

# Test prompt
test_prompt = "Write a short, friendly email about prompt engineering."

# Headers and payload
headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

data = {
    "messages": [
        {"role": "user", "content": test_prompt}
    ],
    "max_completion_tokens": 2500,
    #for gpt 4o and 4.1:  "max_tokens": 100,
    #"temperature": 0.7,
}

# Send the request
response = requests.post(
    f"{endpoint}/openai/deployments/{deployment_id}/chat/completions?api-version={api_version}",
    headers=headers,
    json=data
)

# Show the result
try:
    result = response.json()
    if "choices" in result:
        print("IT WORKED! Here's the response:")
        print(result["choices"][0]["message"]["content"])
    else:
        print("Oops, something went wrong:")
        print(result)
except Exception as e:
    print("Failed to parse JSON response:", e)
    print("Raw response:", response.text)


IT WORKED! Here's the response:
Subject: Exploring the World of Prompt Engineering!

Hi there,

I hope you're doing well! I wanted to share a quick note about prompt engineering—a fascinating field that's all about crafting effective prompts for AI systems. By designing clear, specific instructions, we can guide AI to generate better responses and achieve desired outcomes.

If you’re interested, I’d be happy to chat more or share some tips and resources. It’s an exciting area and can make a big difference in how we interact with technology!

Best wishes,  
[Your Name]


# Model 1
## Data modalities: task duration, and dialogue history.

In [ ]:
def build_prompt_full_history(task_type, task_duration, current_question, current_time, qa_history=[]):
    """
    qa_history: list of tuples (question_time, question, answer)
    current_time: float (in seconds)
    """

    # Filter only previous Q&A before current time
    previous_qa = [(t, q, a) for t, q, a in qa_history if t < current_time]

    prompt = f"""You are an instructor standing beside a student who is performing the task: {task_type}.

The total duration of the task is approximately {int(task_duration)} seconds.
"""

    if previous_qa:
        prompt += f"""\nHere is the full conversation history up to {int(current_time)} seconds:\n"""
        for t, q, a in previous_qa:
            prompt += f"""Student: "{q}" at time {t} seconds \nInstructor: "{a}"\n\n"""
    else:
        prompt += "\nThere has been no prior student question so far. This is the first question.\n"

    prompt += f"""Now, the student asks (at {int(current_time)} seconds):\n Student: "{current_question}"\n\n"""
    prompt += "Please respond like a real-time instructor: clear, concise, short and based on the context that you have.\n"
    
    return prompt

# Model 2
## Data modalities: task duration, task steps, and dialogue history.

In [ ]:
def build_prompt_full_history(task_type, task_duration, current_question, current_time, qa_history=[], task_description=""):
    """
    qa_history: list of tuples (question_time, question, answer)
    current_time: float (in seconds)
    task_description: str, describing the typical steps involved in the task
    """

    # Filter only previous Q&A before current time
    previous_qa = [(t, q, a) for t, q, a in qa_history if t < current_time]

    prompt = f"""You are an instructor standing beside a student who is performing the task: {task_type}.

This task usually follows these common steps (though the exact sequence may vary slightly across different sessions):
{task_description}
\n
The total duration of the task is approximately {int(task_duration)} seconds.
"""

    if previous_qa:
        prompt += f"""\nHere is the full conversation history up to {int(current_time)} seconds:\n"""
        for t, q, a in previous_qa:
            prompt += f"""Student: "{q}" at time {t} seconds \nInstructor: "{a}"\n\n"""
    else:
        prompt += "\n\nThere has been no prior student question so far. This is the first question.\n"

    prompt += f"""Now, the student asks (at {int(current_time)} seconds):\nStudent: "{current_question}"\n\n"""
    prompt += "Please respond like a real-time instructor: clear, concise, short and based on the context that you have.\n"

    return prompt


# Model 3
## Data modalities: task duration, task steps, current task step, and dialogue history.

In [ ]:
def build_prompt_full_history(task_type, task_duration, current_question, current_time, task_step, qa_history=[], task_description=""):
    """
    qa_history: list of tuples (question_time, question, answer)
    current_time: float (in seconds)
    task_description: str, describing the typical steps involved in the task
    """

    # Filter only previous Q&A before current time
    previous_qa = [(t, q, a) for t, q, a in qa_history if t < current_time]

    prompt = f"""You are an instructor standing beside a student who is performing the task: {task_type}.

This task usually follows these common steps (though the exact sequence may vary slightly across different sessions):
{task_description}
\n
The total duration of the task is approximately {int(task_duration)} seconds.
"""

    if previous_qa:
        prompt += f"""\nHere is the full conversation history up to {int(current_time)} seconds:\n"""
        for t, q, a in previous_qa:
            prompt += f"""Student: "{q}" at time {t} seconds \nInstructor: "{a}"\n\n"""
    else:
        prompt += "\n\nThere has been no prior student question so far. This is the first question.\n"

    if task_step == "No coarse-grained action":
        prompt += f"""The current step of the task is not specified. It may be that the student has not yet started the task, or the step detection model may have failed to identify the current step."\n"""

    else:
        prompt += f"""The current step of the task is ({task_step})"\n"""

    prompt += f"""Now, the student asks (at {int(current_time)} seconds):\nStudent: "{current_question}"\n\n"""
    prompt += "Please respond like a real-time instructor: clear, concise, short and based on the context that you have.\n"

    return prompt


# Model 4
## Data modalities: task duration, task steps, current task step, hand action data, and dialogue history for fully context-aware guidance.

In [8]:
def build_prompt_full_history(task_type, task_duration, current_question, current_time, task_step, qa_history=[], task_description="", hand_actions=None, hand_actions_now=None):
    """
    hand_actions: list of [verb, noun, task_step]
    qa_history: list of tuples (question_time, question, answer)
    current_time: float (in seconds)
    task_description: str, describing the typical steps involved in the task
    """

    # Filter Q&A before current time
    previous_qa = [(t, q, a) for t, q, a in qa_history if t < current_time]

    prompt = f"""You are an instructor standing beside a student who is performing the task: {task_type}.

This task usually follows these common steps (though the exact sequence may vary slightly across different sessions):
{task_description}

The total duration of the task is approximately {int(task_duration)} seconds.
"""

    if previous_qa:
        prompt += f"""\nHere is the full conversation history up to {int(current_time)} seconds:\n"""
        for t, q, a in previous_qa:
            prompt += f"""Student: "{q}" at time {t} seconds \nInstructor: "{a}"\n\n"""
    else:
        prompt += "\n\nThere has been no prior student question so far. This is the first question.\n"

    if task_step == "No coarse-grained action":
        prompt += "The current step of the task is not specified. It may be that the student has not yet started the task, or the step detection model may have failed to identify the current step.\n"
    else:
        prompt += f"The current step of the task is: {task_step}\n"

    # Add hand actions (grouped by step)
    
    from collections import defaultdict
    step_to_actions = defaultdict(list)


    raw_data = hand_actions
    if isinstance(raw_data, str):
        hand_actions = ast.literal_eval(raw_data)
    else:
        hand_actions = raw_data  # already parsed

    
    if len(hand_actions) == 0:
        prompt += f"\nRecent hand actions up to last question: No hand actions detected so far.\n"

    else:
        for action in list(hand_actions):
           
            if isinstance(action, (list, tuple)) and len(action) == 3:
                verb, noun, step = action
                
                step_to_actions[step].append(f"{verb} {noun}, ")

        prompt += f"\nRecent hand actions up to the last student question:\n"
        prompt += "(Each action is detected per frame in the form of a verb–noun pair and was extracted using a state-of-the-art hand action recognition model.)\n"
       
        for idx, (step, actions) in enumerate(step_to_actions.items(), 1):
            # if step == "No coarse-grained action":
            #     step = "This step of the task is not specified"
            
            actions_str = ' '.join(actions)
            prompt += f"{actions_str}"

    step_to_actions = defaultdict(list)
    raw_data = hand_actions_now
    if isinstance(raw_data, str):
        hand_actions_now = ast.literal_eval(raw_data)
    else:
        hand_actions_now = raw_data  # already parsed

    
    if len(hand_actions_now) == 0:
        prompt += f"\nMost Recent hand actions after last question (if any) was asked up to {int(current_time)} seconds (time of the current question): No hand actions detected after last question so far.\n"

    else:
        for action in list(hand_actions_now):
           
            if isinstance(action, (list, tuple)) and len(action) == 3:
                verb, noun, step = action
                
                step_to_actions[step].append(f"{verb} {noun}, ")

        prompt += f"\nMost Recent hand actions after last question was asked up to {int(current_time)} seconds (time of the current question):\n"
        # prompt += "(Each action is detected per frame in the form of a verb–noun pair and was extracted using a state-of-the-art hand action recognition model.)\n"
       
        for idx, (step, actions) in enumerate(step_to_actions.items(), 1):
            # if step == "No coarse-grained action":
            #     step = "This step of the task is not specified"
            
            actions_str = ' '.join(actions)
            prompt += f"{actions_str}"


        

    prompt += f"""\n
Now, the student asks (at {int(current_time)} seconds):  
Student: "{current_question}"  

Please respond like a real-time instructor: clear, concise, short, and based on the context you have. Pay close attention to the hand actions data, as it offers real-time insights leading up to the moment the question was asked.
"""
    return prompt


In [ ]:
df = pd.read_csv('Sample Data.csv')
df.head()

,video_name,task_type,task_duration,task_description,task_step,hand_actions_data,hints_before_question,question,question_time,instructor_answer
0,z198-sep-18-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,No coarse-grained action,"[[""grab"", ""dslr"", ""The student detaches the le...","[[1.264, ""We will now operate the DSLR, and th...",Could you repeat please?,58.329,The next step is to detach the lens cover.
1,z198-sep-18-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,No coarse-grained action,"[[""grab"", ""sd_slot"", ""The student detaches the...","[[69.1, ""Nice.""], [69.975, ""Now please attach ...",I'm sorry?,159.200,You can now stop the recording.
2,z073-july-01-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,The student detaches the lens of the camera.,"[[""grab"", ""dslr"", ""The student detaches the le...","[[0.887, ""Now we are going to look at the DSLR...",To detach what?,16.300,To detach the lens you should press on the but...
3,z073-july-01-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,The students changes the battery of the dslr.,"[[""turn"", ""dslr"", ""The student detaches the le...","[[27.2, ""The lens is um, the long round thing,...",Change the battery?,140.100,Yeah.
4,R159-3Oct-RAM,assemble computer,223,The task involves assembling or replacing comp...,No coarse-grained action,[],"[[6.153, ""So the first step is to replace the ...",To replace the RAM card?,10.200,Yep.


In [ ]:
''' 

==========================================
# Generate AI Responses for Each Question
# ------------------------------------------
# This cell loops through the dataset rows and sends prompts 
# to the GPT API using contextual information (task description, current task step, hand actions data, dialouge history).
# 
# Steps:
#   1. Gather previous Q&A and hand actions for the same video.
#   2. Build the prompt via build_prompt_full_history() (for each model, you need to comment the rest).
#   3. Send the prompt to the LLM endpoint.
#   4. Save the AI’s response in df['response_from_ai'].
#   5. Log rows with missing or malformed responses.
# ==========================================


'''

no_answers_rows = []

headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

# Initialize column if not present
if 'response_from_ai' not in df.columns:
    df['response_from_ai'] = ""

for idx in tqdm(df.index):
    if idx % 50 == 0:
        print(f"Processing row {idx}...")

    row = df.loc[idx]
    video = row['video_name']
    current_time = row['question_time']

    if pd.isna(current_time):
        continue  # skip if no time info

    # Get all previous Q&A for this video before this time
    earlier_rows = df[
        (df['video_name'] == video) &
        (df['question_time'] < current_time)
    ]

    # Use already filled responses
    qa_history = list(zip(
        earlier_rows['question_time'],
        earlier_rows['question'],
        earlier_rows['response_from_ai']
    ))

    # Step 1: Collect all previous hand actions into a single list
    all_prev_hand_actions = []
    for _, r in earlier_rows.iterrows():
        if not pd.isna(r['hand_actions_data']):
            try:
                ha = ast.literal_eval(r['hand_actions_data']) if isinstance(r['hand_actions_data'], str) else r['hand_actions_data']
                all_prev_hand_actions.extend(ha)
            except Exception as e:
                print(f"❌ Error parsing hand actions at row {r.name}: {e}")


    # Step 2: Parse current row's hand actions
    hand_actions_now = []
    if not pd.isna(row['hand_actions_data']):
        try:
            hand_actions_now = ast.literal_eval(row['hand_actions_data']) if isinstance(row['hand_actions_data'], str) else row['hand_actions_data']
        except Exception as e:
            print(f"❌ Error parsing current hand actions at row {idx}: {e}")
            hand_actions_now = []

    # Build prompt
    prompt = build_prompt_full_history(
        task_type=row['task_type'],
        task_duration=row['task_duration'],
        current_question=row['question'],
        current_time=current_time,
        task_step = row['task_step'],
        qa_history=qa_history,
        task_description = row['task_description'],
        hand_actions= all_prev_hand_actions,
        hand_actions_now = hand_actions_now
    )

    # print(prompt)
    # print("++++++++++++++++++++++++++++++++")

    # Send to GPT API
    data = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        # "max_completion_tokens": 2500,
        "temperature": 0.2,
        "max_tokens": 100
    }

    try:
        response = requests.post(
            f"{endpoint}/openai/deployments/{deployment_id}/chat/completions?api-version={api_version}",
            headers=headers,
            json=data
        )

        result = response.json()
        if ("choices" in result and 
            "message" in result["choices"][0] and 
            "content" in result["choices"][0]["message"]):

            ai_reply = result["choices"][0]["message"]["content"]
            ai_reply = re.sub(r'^\s*Instructor\s*:\s*', '', ai_reply, flags=re.IGNORECASE)
            df.at[idx, "response_from_ai"] = ai_reply

        else:
            print(f"⚠️ Unexpected API response at row {idx}:\n{result}")
            df.at[idx, "response_from_ai"] = "[Error: Unexpected structure]"

            no_answers_rows.append(idx)

            # Save malformed result to a JSON file
            # with open(f"failed_response_row_{idx}.json", "w") as f:
            #     json.dump(result, f, indent=2)


    except Exception as e:
        print(f"❌ Error at row {idx}: {e}")
        df.at[idx, "response_from_ai"] = "[Error: API call failed]"


  0%|          | 0/10 [00:00<?, ?it/s]

Processing row 0...


100%|██████████| 10/10 [00:11<00:00,  1.12s/it]


In [ ]:
''' Collect responses from all models in distinct columns (Model 1, Model 2, Model 3, Model 4) to enable comparative evaluation using multiple methods, 
including lexical comparison and LLM-as-a-Judge. (Evaluation Methods.ipynb) '''

' Collect responses from all models in distinct columns (Model 1, Model 2, Model 3, Model 4) to enable comparative evaluation using multiple methods, including lexical comparison and LLM-as-a-Judge. (Evaluation Methods.ipynb)'

In [ ]:
df.to_csv('Sample Data_responses.csv')